# AutoRA Workflow

Generated by AutoRA Workflow Editor on 2026-06-15T22:32:47.875Z

## 1. Install dependencies

In [ ]:
%pip install autora-synthetic==1.0.0 autora-experimentalist-bandit-random==1.0.0 autora-theorist-bsr==1.0.0

## 2. Imports

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List

from autora.state import on_state, Delta, estimator_on_state, StandardState
from autora.variable import VariableCollection, Variable
from autora.experimentalist.bandit_random import bandit_random_pool
from autora.experimentalist.random import random_sample
from autora.experiment_runner.synthetic.economics.expected_value_theory import expected_value_theory
from autora.theorist.bsr.regressor import BSRRegressor

import pandas as pd
import numpy as np

## 3. Component definitions

In [ ]:
# Bandit Random Pooler
@on_state()
def bandit_random_pooler_on_state(variables: VariableCollection) -> Delta:
    return Delta(conditions=bandit_random_pool(variables, num_samples=1))

In [ ]:
# Random Sample
@on_state()
def random_sample_on_state(conditions: pd.DataFrame, num_samples: int = 1) -> Delta:
    return Delta(conditions=random_sample(conditions=conditions, num_samples=num_samples, replace=False))

In [ ]:
# Expected Value Theory (Synthetic, Economics)
@on_state()
def expected_value_theory_synthetic_economics_on_state(conditions: pd.DataFrame) -> Delta:
    runner = expected_value_theory(choice_temperature=0.1, value_lambda=0.5, resolution=10, minimum_value=-1, maximum_value=1)
    assert runner.run is not None
    return Delta(experiment_data=runner.run(conditions=conditions))

In [ ]:
# BSR Regressor
bsr_regressor_on_state = estimator_on_state(BSRRegressor(tree_num=3, itr_num=5000, alpha1=0.4, alpha2=0.4, beta=-1, show_log=False, val=100, last_idx=-1, prior_name="Uniform"))

## 4. Run the workflow

In [ ]:
# Initialize variables - customize this based on your experiment
# You may need to define your own variables or get them from an experiment runner
variables = VariableCollection(
    independent_variables=[
        Variable(name="x", allowed_values=np.linspace(-1, 1, 100))
    ],
    dependent_variables=[
        Variable(name="y")
    ]
)

# Initialize state
state = StandardState(variables=variables)

# Main experiment loop (1 cycles)
for i in range(1):
    print(f'Cycle {i}')

    # Bandit Random Pooler
    state = bandit_random_pooler_on_state(state)

    # Random Sample
    state = random_sample_on_state(state, num_samples=1)

    # Expected Value Theory (Synthetic, Economics)
    state = expected_value_theory_synthetic_economics_on_state(state)

    # BSR Regressor
    state = bsr_regressor_on_state(state)


print("Workflow completed!")
state